# AI usage

# Description

# Links
- GitHub repository: https://github.com/xoksy/jouyit_IND320 
- streamlit app: https://jouyitind320-fafhfv6wkn6ecfeejafjgj.streamlit.app/

In [ ]:
import geopandas as gpd
import pandas as pd

# Read the shapefile for precipitation regions
precipitation_regions = gpd.read_file("../NVE Atlas/NVE_Region.shp", encoding="utf-8")
precipitation_regions

,OBJECTID,REGION,REGIONNAVN,POSTNUMMER,POSTSTED,BESØKSADR,ADRESSE,geometry
0,1.0,RV,Region Vest,6801,Førde,Naustdalsveien 1B,Postboks 53,"POLYGON ((193105.76 6929797.839, 191705.75 692..."
1,2.0,RM,Region Midt-Norge,7030,Trondheim,"Abelsgate 9, Trondheim",Abelsgate 9,"POLYGON ((391250 7388620, 392270 7386790, 3937..."
2,3.0,RN,Region Nord,8514,Narvik,Kongensgate 52-54,Kongensgate 52-54,"POLYGON ((1109096.413 7844647.502, 1105115.343..."
3,4.0,RØ,Region Øst,2307,Hamar,Vangsvegen 73,Postboks 4223,"POLYGON ((253748.22 6678130.91, 253645.58 6678..."
4,5.0,RS,Region Sør,3103,Tønsberg,Anton Jenssens gate 7,Postboks 2124,"POLYGON ((253748.22 6678130.91, 253917.63 6677..."


In [14]:
#1000 points per region
points_per_region = precipitation_regions['geometry'].sample_points(size=1000, random_state=42)

#Convert to gps coordinates (show in the course)
points_wgs84 = points_per_region.to_crs(epsg=4326)

#Store the points in a list of dictionaries
points_precipitation = []

for idx, multipoint in points_wgs84.items():
    region_navn = precipitation_regions.loc[idx,'REGION']
    for point_id, point in enumerate(multipoint.geoms):
        points_precipitation.append({
            "region": region_navn,
            "point_id": point_id,
            "latitude": point.y,
            "longitude": point.x
        })

# Create a DataFrame
precipitation_df = pd.DataFrame(points_precipitation)

# Display the first rows
print(precipitation_df.head())

# Check the number of points
print(f"Total number of points: {len(precipitation_df)}")

  region  point_id   latitude  longitude
0     RV         0  61.909669   6.573643
1     RV         1  61.867092   5.117018
2     RV         2  59.683949   5.415274
3     RV         3  60.462839   7.490924
4     RV         4  62.680823   8.204631
Total number of points: 5000


In [17]:
#Same for electricity regions
electricity_regions = gpd.read_file("../NVE Atlas/Budområder.shp", encoding="utf-8")

#1000 points per region
points_per_region = electricity_regions['geometry'].sample_points(size=1000, random_state=42)

#Convert to gps coordinates (show in the course)
points_wgs84 = points_per_region.to_crs(epsg=4326)

#Store the points in a list of dictionaries
points_electricity = []

for idx, multipoint in points_wgs84.items():
    region_navn = electricity_regions.loc[idx,'budomr']
    for point_id, point in enumerate(multipoint.geoms):
        points_electricity.append({
            "zone": region_navn,
            "point_id": point_id,
            "latitude": point.y,
            "longitude": point.x
        })

# Create a DataFrame
electricity_df = pd.DataFrame(points_electricity)

# Display the first rows
print(electricity_df.head())

# Check the number of points
print(f"Total number of points: {len(electricity_df)}")

   zone  point_id   latitude  longitude
0  NO 1         0  59.227426  10.669684
1  NO 1         1  61.106296   8.182644
2  NO 1         2  62.730356  11.616917
3  NO 1         3  61.765191  11.926571
4  NO 1         4  61.855655  11.716801
Total number of points: 5000


In [ ]:
import requests
import pandas as pd
import time
from pathlib import Path

OPEN_METEO_URL = "https://archive-api.open-meteo.com/v1/archive"


def download_weather_data(
    region_id: str,
    start_date: str,
    end_date: str,
    locations: pd.DataFrame,
    start_from: int = 0,
    output_file: str = "weather_NO3.csv",
) -> pd.DataFrame:

    # Normalize zone names
    zone_names = (
        locations["zone"]
        .astype(str)
        .str.replace(" ", "", regex=False)
        .str.upper()
    )

    region_locations = locations[
        zone_names == region_id.replace(" ", "").upper()
    ].reset_index(drop=True)

    if region_locations.empty:
        raise ValueError(f"No locations found for zone {region_id}.")

    output_path = Path(output_file)
    file_exists = output_path.exists()

    with requests.Session() as session:

        for position in range(start_from, len(region_locations)):

            location = region_locations.iloc[position]

            params = {
                "latitude": float(location["latitude"]),
                "longitude": float(location["longitude"]),
                "start_date": start_date,
                "end_date": end_date,
                "hourly": (
                    "temperature_2m,"
                    "precipitation,"
                ),
                "models": "era5",
                "timezone": "Europe/Oslo",
            }

            for attempt in range(6):
                try:
                    response = session.get(
                        OPEN_METEO_URL,
                        params=params,
                        timeout=120,
                    )

                    if response.status_code == 429:
                        retry_after = response.headers.get("Retry-After")

                        if retry_after and retry_after.isdigit():
                            wait_seconds = int(retry_after)
                        else:
                            wait_seconds = min(60 * (2 ** attempt), 1800)

                        print(
                            f"Rate limit at point {position}. "
                            f"Waiting {wait_seconds}s "
                            f"(attempt {attempt + 1}/6)."
                        )

                        time.sleep(wait_seconds)
                        continue

                    response.raise_for_status()
                    result = response.json()

                    if "hourly" not in result:
                        raise ValueError(
                            f"No hourly data returned for point {position}."
                        )

                    weather_df = pd.DataFrame(result["hourly"])

                    weather_df["region"] = (
                        region_id.replace(" ", "").upper()
                    )
                    weather_df["point_id"] = location["point_id"]
                    weather_df["latitude"] = location["latitude"]
                    weather_df["longitude"] = location["longitude"]

                    # Save each point immediately
                    weather_df.to_csv(
                        output_path,
                        mode="a",
                        header=not file_exists,
                        index=False,
                    )
                    file_exists = True

                    break

                except requests.RequestException as error:
                    if attempt == 5:
                        raise RuntimeError(
                            f"Failed at point {position}. "
                            f"Resume from start_from={position} "
                            f"after checking the API limits."
                        ) from error

                    wait_seconds = min(30 * (2 ** attempt), 1800)
                    print(
                        f"Request failed: {error}. "
                        f"Waiting {wait_seconds}s."
                    )
                    time.sleep(wait_seconds)

            else:
                raise RuntimeError(
                    f"Too many 429 responses at point {position}. "
                    f"Stop and check the API limits before retrying."
                )

            if (position + 1) % 10 == 0:
                print(
                    f"Processed {position + 1}/"
                    f"{len(region_locations)} points."
                )

            # Conservative delay between successful requests
            time.sleep(2)

    return pd.read_csv(output_path)

In [ ]:
weather_df = download_weather_data(
    region_id="NO3",
    start_date="2024-01-01",
    end_date="2024-12-31",
    locations=electricity_df,
    start_from=0,
    output_file="weather_NO3.csv",
)

print(weather_df.head())
print(weather_df.shape)

HTTPError: 429 Client Error: Too Many Requests for url: https://archive-api.open-meteo.com/v1/archive?latitude=63.6543981786748&longitude=10.706520708982099&start_date=2024-01-01&end_date=2024-12-31&hourly=temperature_2m%2Cprecipitation%2Crelative_humidity_2m%2Cwind_speed_10m&models=era5&timezone=Europe%2FOslo